# Notebook 02 — Silver: Data Cleaning

**What we are doing:** reading the 75 valid records from Bronze, trimming and standardizing text columns (`product_name`, `brand`, `store_city`), casting datatypes (date, int, double), calculating `net_sales = (quantity × unit_price) − discount`, and removing duplicate `sale_id`s so each sale appears exactly once.

**Flow:** Read valid Bronze records → Trim and standardize columns → Cast datatypes → Calculate net_sales → Remove duplicate sale IDs

**Output — Clean Silver Delta table:**

* `electronic_sales.silver.electronics_sales` (primary — Gold reads this) + `db_final.silver.electronics_sales` (mirror in the db_final catalog).

**Cell 1 — Build silver table:** trims `product_name`, uppercases `brand`/`store_city`, casts date & numerics, `net_sales = (quantity × unit_price) − discount`, one row per unique `sale_id` → `electronic_sales.silver.electronics_sales` + `db_final.silver.electronics_sales`.

In [0]:
from pyspark.sql.functions import *

valid_df = spark.read.table("electronic_sales.bronze.electronics_sales_valid")

silver_df = valid_df \
    .withColumn("product_name", trim(col("product_name"))) \
    .withColumn("brand", upper(col("brand"))) \
    .withColumn("store_city", upper(col("store_city"))) \
    .withColumn("sale_date", to_date(col("sale_date"))) \
    .withColumn("quantity", col("quantity").cast("int")) \
    .withColumn("unit_price", col("unit_price").cast("double")) \
    .withColumn("discount", col("discount").cast("double")) \
    .withColumn("warranty_years", col("warranty_years").cast("int")) \
    .withColumn(
        "net_sales",
        (col("quantity") * col("unit_price")) - col("discount")
    ) \
    .dropDuplicates(["sale_id"])

silver_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "electronic_sales.silver.electronics_sales"
    )

silver_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "db_final.silver.electronics_sales"
    )

display(silver_df)

**Silver layer complete:** 75 valid rows in = 75 silver rows out, one row per `sale_id`, `net_sales` verified with 0 mismatches ✅